In [31]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding Model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4954.79it/s]


Embedding Model loaded


In [32]:
documents = ["Employees receive 30 days of annual leavel",
             "International business travel allows hotel expenses up to 700 AED per night",
             "Employees receive 60 days of maternity leave.",
             "Medical insurance covers employees and their dependents",
             "Work from home is permitted twice per week"
             ]
print("Number of documents:",len(documents))

Number of documents: 5


In [33]:
document_embeddings = model.encode(documents)
print(document_embeddings.shape)
print(document_embeddings[0])

(5, 384)
[ 4.14817594e-03  4.52192575e-02  3.68318781e-02  2.08983794e-02
  3.47575964e-03  7.25992918e-02 -1.32553885e-02 -5.82575426e-02
 -4.54212539e-02 -5.39189670e-03  8.23232830e-02 -1.58660877e-02
 -3.19534056e-02 -2.79682013e-03  1.15142260e-02  3.26208286e-02
 -4.25304770e-02  9.34563577e-03  3.79903801e-02 -7.36751482e-02
 -4.70963158e-02 -5.92467524e-02 -7.22874254e-02  2.31252685e-02
  2.75986716e-02  3.65090780e-02 -3.20540927e-02  1.78036317e-02
 -3.53646651e-02 -2.68605887e-03 -4.27014800e-03  2.35243123e-02
 -5.15788514e-03 -1.39624896e-02 -1.44132785e-02 -1.00736190e-02
 -8.25974420e-02 -1.58202369e-02  1.99463237e-02  2.27567758e-02
 -6.40818179e-02 -1.45295504e-02  3.48132066e-02 -1.68974679e-02
 -8.87558162e-02 -3.65344658e-02 -4.66618687e-02  4.08666469e-02
  2.63391770e-02  1.62189588e-01  8.72365162e-02  2.28935760e-02
 -2.37767696e-02  9.19775441e-02 -5.19979149e-02 -1.79101229e-02
  5.26196249e-02 -1.12180293e-01 -1.69022474e-02  2.09090319e-02
 -5.08615142e-03

In [34]:
query = "How many days of maternity leave do employees get?"
query_embedding = model.encode(query)
print(query_embedding.shape)

(384,)


In [35]:
from sentence_transformers.util import cos_sim

similarities = cos_sim(query_embedding, document_embeddings)
print(similarities)

tensor([[0.6465, 0.0938, 0.8647, 0.3038, 0.2915]])


In [36]:
top_k = 2
top_results = similarities[0].topk(top_k)
print(top_results)

torch.return_types.topk(
values=tensor([0.8647, 0.6465]),
indices=tensor([2, 0]))


In [37]:
for index in top_results.indices:
    print(documents[index])


Employees receive 60 days of maternity leave.
Employees receive 30 days of annual leavel


In [38]:
for index in top_results.indices:
    print(documents[index])

Employees receive 60 days of maternity leave.
Employees receive 30 days of annual leavel


In [39]:
retrieved_context = ""

for index in top_results.indices:
    retrieved_context +=documents[index] + "\n"

print(retrieved_context)

Employees receive 60 days of maternity leave.
Employees receive 30 days of annual leavel



In [40]:
prompt = f"""
Use the following context to answer the question.

Context:
{retrieved_context}

Question:
{query}

Answer using only the provided context.
"""

print(prompt)


Use the following context to answer the question.

Context:
Employees receive 60 days of maternity leave.
Employees receive 30 days of annual leavel


Question:
How many days of maternity leave do employees get?

Answer using only the provided context.



In [41]:
from openai import OpenAI
client = OpenAI()
print("OpenAI client ready!")

OpenAI client ready!


In [42]:
import os

print("API key available:", bool(os.getenv("OPENAI_API_KEY")))

API key available: True


In [43]:
response = client.responses.create(
    model = "gpt-5.6",
    input=prompt
)
print(response.output_text)

Employees receive 60 days of maternity leave.


In [44]:
import chromadb

client = chromadb.Client()

print("Chroma Client created")

Chroma Client created


In [45]:
collection = client.get_or_create_collection(name = "employee_policies")

print("Collection ready")

Collection ready


In [46]:
first_document = documents[0]
first_embedding  = document_embeddings[0]

collection.add(
    ids=["doc1"],
    documents=[first_document],
    embeddings=[first_embedding.tolist()]
)

print("First document added")

First document added


In [47]:
stored=collection.get(
    ids=["doc1"],
    include=["documents","embeddings"]
)
print(stored)

{'ids': ['doc1'], 'embeddings': array([[ 4.14817594e-03,  4.52192575e-02,  3.68318781e-02,
         2.08983794e-02,  3.47575964e-03,  7.25992918e-02,
        -1.32553885e-02, -5.82575426e-02, -4.54212539e-02,
        -5.39189670e-03,  8.23232830e-02, -1.58660877e-02,
        -3.19534056e-02, -2.79682013e-03,  1.15142260e-02,
         3.26208286e-02, -4.25304770e-02,  9.34563577e-03,
         3.79903801e-02, -7.36751482e-02, -4.70963158e-02,
        -5.92467524e-02, -7.22874254e-02,  2.31252685e-02,
         2.75986716e-02,  3.65090780e-02, -3.20540927e-02,
         1.78036317e-02, -3.53646651e-02, -2.68605887e-03,
        -4.27014800e-03,  2.35243123e-02, -5.15788514e-03,
        -1.39624896e-02, -1.44132785e-02, -1.00736190e-02,
        -8.25974420e-02, -1.58202369e-02,  1.99463237e-02,
         2.27567758e-02, -6.40818179e-02, -1.45295504e-02,
         3.48132066e-02, -1.68974679e-02, -8.87558162e-02,
        -3.65344658e-02, -4.66618687e-02,  4.08666469e-02,
         2.63391770e-02,

In [48]:
print("ID",stored["ids"][0])
print("Document:",stored["documents"][0])
print("Embedding dimensions:",len(stored["embeddings"][0]))


ID doc1
Document: Employees receive 30 days of annual leavel
Embedding dimensions: 384


In [49]:
collection.add(
    ids=["doc2","doc3","doc4","doc5"],
    documents=documents[1:],
    embeddings=document_embeddings[1:].tolist()
)

print("Total documents in chroma:",collection.count())

Total documents in chroma: 5


In [50]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=2
)

print(results)

{'ids': [['doc3', 'doc1']], 'embeddings': None, 'documents': [['Employees receive 60 days of maternity leave.', 'Employees receive 30 days of annual leavel']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.27066949009895325, 0.7069528102874756]]}


In [51]:
for doc_id, document,distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]

):
    print("ID",doc_id)
    print("Document:",document)
    print("Distance:",distance)
    print()

ID doc3
Document: Employees receive 60 days of maternity leave.
Distance: 0.27066949009895325

ID doc1
Document: Employees receive 30 days of annual leavel
Distance: 0.7069528102874756



In [52]:
cosine_collection = client.get_or_create_collection(
    name="employee_policies_cosine",
    metadata={"hnsw:space":"cosine"}
)
print("Cosine collection created")

Cosine collection created


In [53]:
cosine_collection.add(
    ids=["doc1","doc2","doc3","doc4","doc5"],
    documents=documents,
    embeddings=document_embeddings.tolist()
)
print("Total Documents in cosine collection:",cosine_collection.count())


Total Documents in cosine collection: 5


In [54]:
cosine_results = cosine_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=2
)

print(cosine_results["ids"])
print(cosine_results["documents"])
print(cosine_results["distances"])


[['doc3', 'doc1']]
[['Employees receive 60 days of maternity leave.', 'Employees receive 30 days of annual leavel']]
[[0.1353345513343811, 0.3534764051437378]]


In [56]:
from sentence_transformers.util import cos_sim

manual_check = cos_sim(query_embedding,document_embeddings)
print(manual_check)

tensor([[0.6465, 0.0938, 0.8647, 0.3038, 0.2915]])


In [57]:
print("Doc3 manual cosine:", manual_check[0][2].item())
print("Doc1 manual cosine:", manual_check[0][0].item())

print("Doc3 expected cosine distance:", 1 - manual_check[0][2].item())
print("Doc1 expected cosine distance:", 1 - manual_check[0][0].item())

Doc3 manual cosine: 0.8646652698516846
Doc1 manual cosine: 0.6465235948562622
Doc3 expected cosine distance: 0.13533473014831543
Doc1 expected cosine distance: 0.3534764051437378
